# Analiza napadalnih igralcev lige NFL v rednem delu sezone 2025/26

Analizirali bomo podatke o NFL igralcih iz sezone 2025/26, pridobljene s portala [Pro-Football-Reference](https://www.pro-football-reference.com). Zajeti so podatki o podajah (Passing), tekih (Rushing), ujetih podajah (Receiving) in obrambi (Defense).

#### Priprava podatkov

Najprej uvozimo vse potrebne knjižnice ter pripravimo podatke za analizo. Številske stolpce pretvorimo v ustrezne tipe in ustvarimo dodatne metrike za analizo učinkovitosti igralcev.

In [37]:
import pandas as pd
import matplotlib.pyplot as plt

# Nastavitev za lepši izris grafov
%matplotlib inline

# Naložimo CSV datoteko
df = pd.read_csv("podatki/nfl_2025_statistika.csv")

st_vrstic = len(df)
st_stolpcev = len(df.columns)

print(f"Spodaj lahko vidimo tabelo NFL igralcev z vsemi stolpci, ki jih bomo uporabljali. Imamo tabelo {st_vrstic} igralcev z {st_stolpcev} stolpci: med njimi so ime igralca, ekipa, pozicija, starost, odigrane tekme ter zbrane statistike po kategorijah.")

display(df)

Spodaj lahko vidimo tabelo NFL igralcev z vsemi stolpci, ki jih bomo uporabljali. Imamo tabelo 1764 igralcev z 21 stolpci: med njimi so ime igralca, ekipa, pozicija, starost, odigrane tekme ter zbrane statistike po kategorijah.


,igralec,ekipa,pozicija,starost,tekme,passing_yards,passing_td,passing_att,passing_cmp,passing_int,...,rushing_yards,rushing_td,rushing_att,fumbles,receiving_yards,receiving_td,receptions,targets,def_interceptions,def_sacks
0,A'Shawn Robinson,CAR,DT,30,17,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A.J. Brown,PHI,WR,28,15,0,0,0,0,0,...,0,0,0,0,1028,7,81,128,0,0
2,A.J. Epenesa,BUF,DE,27,16,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
3,A.J. Green,MIA,CB,27,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AJ Barner,SEA,TE,23,17,0,0,0,0,0,...,16,1,11,6,586,7,58,75,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1759,Zeek Biggers,MIA,DT,22,9,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1760,Zion Childress,DAL,CB,23,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1761,Zion Logue,BUF,DT,24,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1762,Zonovan Knight,ARI,RB,24,12,0,0,0,0,0,...,269,4,82,2,160,1,22,31,0,0


#### Pregled najboljših igralcev

Spodaj je prikazana tabela 10 najučinkovitejših igralcev lige glede na skupno število pridobljenih jardov (podaje + teki + sprejemi) v sezoni. Tabela vključuje tudi podrobnejše podatke pridobljenih jardov po posameznih kategorijah ter skupno število doseženih touchdownov (`total_td`). Pričakovano so vsi igralci podajalci (quarterbacki), razumljivo pa je nizko oziroma ničelno število jardov po sprejemu (receiving yards), saj je to naloga sprejemalcev (receivers) in podajalci priložnosti za lovljenje nimajo razen ob "trick plays", ko žogo meče nekdo, ki ni podajalec in napadalna ekipa želi zmesti obrambno.

In [51]:
import pandas as pd

# Naložimo podatke
df = pd.read_csv("podatki/nfl_2025_statistika.csv")

# Izračunamo skupne jarde in TD-je, če še niso izračunani
df["skupni_jardi"] = df.get("passing_yards", 0) + df.get("rushing_yards", 0) + df.get("receiving_yards", 0)
df["skupni_td"] = df.get("passing_td", 0) + df.get("rushing_td", 0) + df.get("receiving_td", 0)

# Pretvorimo vse številske stolpce v cela števila (brez .0)
for col in df.select_dtypes(include=["float64", "int64"]).columns:
    df[col] = df[col].fillna(0).astype(int)

# Pripravimo top 10
top_10_vsi = df.sort_values(by="skupni_jardi", ascending=False).head(10).copy()
top_10_vsi = top_10_vsi.reset_index(drop=True)
top_10_vsi.index = top_10_vsi.index + 1

# Prikaz tabele v Jupyter Notebooku
top_10_vsi[["igralec", "ekipa", "pozicija", "tekme", "passing_yards", "rushing_yards", "receiving_yards", "skupni_jardi", "skupni_td"]]

,igralec,ekipa,pozicija,tekme,passing_yards,rushing_yards,receiving_yards,skupni_jardi,skupni_td
1,Drake Maye,NWE,QB,17,4394,450,2,4846,35
2,Dak Prescott,DAL,QB,17,4552,177,0,4729,32
3,Matthew Stafford,LAR,QB,17,4707,1,0,4708,46
4,Jared Goff,DET,QB,17,4564,45,0,4609,34
5,Trevor Lawrence,JAX,QB,17,4007,359,0,4366,38
6,Caleb Williams,CHI,QB,17,3942,388,22,4352,31
7,Bo Nix,DEN,QB,17,3931,356,0,4287,30
8,Josh Allen,BUF,QB,17,3668,579,0,4247,39
9,Justin Herbert,LAC,QB,16,3727,498,0,4225,28
10,Sam Darnold,SEA,QB,17,4048,95,0,4143,25


#### Najboljša razmerja med touchdowni (TD) in prestreženimi podajami (INT)

Spodnja tabela prikazuje podajalce, ki so dosegli vsaj 10 touchdownov in zbrali vsaj 1500 jardov s podajo, razvrščeni glede na učinkovitost (razmereje med TD in INT)

In [55]:
import pandas as pd

df = pd.read_csv("podatki/nfl_2025_statistika.csv")
qb_df = df[df["pozicija"] == "QB"].copy()

# Izračunamo razmerje TD/INT
qb_df["td_int_razmerje"] = qb_df["passing_td"] / qb_df["passing_int"].replace(0, 1)

# Uporabimo pogoja: vsaj 10 TD-jev in vsaj 1500 passing yardov
filtrirani_qb = qb_df[(qb_df["passing_td"] >= 10) & (qb_df["passing_yards"] >= 1500)]
rezultat = filtrirani_qb.sort_values(by="td_int_razmerje", ascending=False).head(10).copy()

# Ponastavimo indeks, da se začne pri 1
rezultat = rezultat.reset_index(drop=True)
rezultat.index = rezultat.index + 1

print("Top QB-ji po razmerju TD/INT (min. 10 TD in min. 1500 yardov):")
rezultat[["igralec", "ekipa", "passing_yards", "passing_td", "passing_int", "td_int_razmerje"]]

Top QB-ji po razmerju TD/INT (min. 10 TD in min. 1500 yardov):


,igralec,ekipa,passing_yards,passing_td,passing_int,td_int_razmerje
1,Matthew Stafford,LAR,4707,46,8,5.750000
2,Jared Goff,DET,4564,34,8,4.250000
3,Jalen Hurts,PHI,3224,25,6,4.166667
4,Drake Maye,NWE,4394,31,8,3.875000
5,Caleb Williams,CHI,3942,27,7,3.857143
6,Jordan Love,GNB,3381,23,6,3.833333
7,Aaron Rodgers,PIT,3322,24,7,3.428571
8,Joe Burrow,CIN,1809,17,5,3.400000
9,Jaxson Dart,NYG,2272,15,5,3.000000
10,Dak Prescott,DAL,4552,30,10,3.000000


#### Primerjava globokih podaj (Y/A) in natančnosti (Cmp%)
Spodnja tabela primerja povprečne osvojene jarde na poskus podaje (Y/A) z odstotkom točnih podaj za podajalce z vsaj 150 poskusi podaj.

In [57]:
# Izračunamo jarde na podajo in odstotek točnih podaj
qb_df["jardi_na_podajo"] = qb_df["passing_yards"] / qb_df["passing_att"].replace(0, 1)
qb_df["procent_točnih"] = (qb_df["passing_cmp"] / qb_df["passing_att"].replace(0, 1)) * 100

# Filtriramo, sortiramo po jardih na podajo in vzamemo top 10
rezultat_ya = qb_df[qb_df["passing_att"] > 150].sort_values(by="jardi_na_podajo", ascending=False).head(10).copy()

# Ponastavimo indeks od 1 do 10
rezultat_ya = rezultat_ya.reset_index(drop=True)
rezultat_ya.index = rezultat_ya.index + 1

print("Primerjava globokih podajalcev (min. 150 poskusov):")
rezultat_ya[["igralec", "ekipa", "procent_točnih", "jardi_na_podajo", "passing_yards"]]

Primerjava globokih podajalcev (min. 150 poskusov):


,igralec,ekipa,procent_točnih,jardi_na_podajo,passing_yards
1,Drake Maye,NWE,71.951220,8.930894,4394
2,Sam Darnold,SEA,67.714885,8.486373,4048
3,Lamar Jackson,BAL,63.576159,8.440397,2549
4,Daniel Jones,IND,67.968750,8.075521,3101
5,Josh Allen,BUF,69.347826,7.973913,3668
6,Jared Goff,DET,67.993080,7.896194,4564
7,Matthew Stafford,LAR,64.991625,7.884422,4707
8,Jordan Love,GNB,66.287016,7.701595,3381
9,Brock Purdy,SFO,69.366197,7.630282,2167
10,Dak Prescott,DAL,67.333333,7.586667,4552
